# V-JEPA 2 Demo Notebook

This tutorial provides an example of how to load the V-JEPA 2 model in vanilla PyTorch and HuggingFace, extract a video embedding, and then predict an action class. For more details about the paper and model weights, please see https://github.com/facebookresearch/vjepa2.

First, let's import the necessary libraries and load the necessary functions for this tutorial.

In [1]:
import json
import os
import subprocess

import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader
from transformers import AutoVideoProcessor, AutoModel

import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms
from src.models.attentive_pooler import AttentiveClassifier
from src.models.vision_transformer import vit_giant_xformers_rope

IMAGENET_DEFAULT_MEAN = (0.485, 0.456, 0.406)
IMAGENET_DEFAULT_STD = (0.229, 0.224, 0.225)


def get_best_gpu():
    """Return the torch.device for the GPU with the most free memory."""
    num_gpus = torch.cuda.device_count()
    if num_gpus == 0:
        raise RuntimeError("No CUDA GPUs available")
    best_idx, best_free = 0, 0
    for i in range(num_gpus):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i} ({torch.cuda.get_device_name(i)}): "
              f"{free / 1024**3:.1f} GiB free / {total / 1024**3:.1f} GiB total")
        if free > best_free:
            best_free = free
            best_idx = i
    device = torch.device(f"cuda:{best_idx}")
    print(f"  -> Selected {device}")
    return device


def load_model_to_best_gpu(model, label="Model"):
    """Move a model to the GPU with the most free memory and set to eval mode."""
    print(f"\nPlacing {label}:")
    device = get_best_gpu()
    model.to(device).eval()
    return device


def load_pretrained_vjepa_pt_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 encoder
    # Load to CPU first to avoid GPU memory spikes, then load_state_dict
    # copies only the needed tensors to the model's device
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["encoder"]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def load_pretrained_vjepa_classifier_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 classifier
    # Load to CPU first to avoid GPU memory spikes
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["classifiers"][0]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def build_pt_video_transform(img_size):
    short_side_size = int(256.0 / 224 * img_size)
    # Eval transform has no random cropping nor flip
    eval_transform = video_transforms.Compose(
        [
            video_transforms.Resize(short_side_size, interpolation="bilinear"),
            video_transforms.CenterCrop(size=(img_size, img_size)),
            volume_transforms.ClipToTensor(),
            video_transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
        ]
    )
    return eval_transform


def get_video():
    vr = VideoReader("sample_video.mp4")
    # choosing some frames here, you can define more complex sampling strategy
    frame_idx = np.arange(0, 128, 2)
    video = vr.get_batch(frame_idx).asnumpy()
    return video


def forward_vjepa_video(model_hf, model_pt, hf_transform, pt_transform):
    # Run a sample inference with VJEPA
    # Automatically detect which device each model is on
    device_hf = next(model_hf.parameters()).device
    device_pt = next(model_pt.parameters()).device
    with torch.inference_mode():
        # Read and pre-process the image
        video = get_video()  # T x H x W x C
        video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
        x_pt = pt_transform(video).to(device_pt).unsqueeze(0)
        x_hf = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to(device_hf)
        # Extract the patch-wise features from the last layer
        out_patch_features_pt = model_pt(x_pt)
        out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features_hf, out_patch_features_pt


def get_vjepa_video_classification_results(classifier, out_patch_features_pt):
    SOMETHING_SOMETHING_V2_CLASSES = json.load(open("ssv2_classes.json", "r"))

    with torch.inference_mode():
        out_classifier = classifier(out_patch_features_pt)

    print(f"Classifier output shape: {out_classifier.shape}")

    print("Top 5 predicted class names:")
    top5_indices = out_classifier.topk(5).indices[0]
    top5_probs = F.softmax(out_classifier.topk(5).values[0]) * 100.0  # convert to percentage
    for idx, prob in zip(top5_indices, top5_probs):
        str_idx = str(idx.item())
        print(f"{SOMETHING_SOMETHING_V2_CLASSES[str_idx]} ({prob}%)")

    return

/home/ubuntu/inwdata/prithvi/git/vjepa2/venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Next, let's download a sample video to the local repository. If the video is already downloaded, the code will skip this step. Likewise, let's download a mapping for the action recognition classes used in Something-Something V2, so we can interpret the predicted action class from our model.

In [2]:
sample_video_path = "sample_video.mp4"
# Download the video if not yet downloaded to local path
if not os.path.exists(sample_video_path):
    video_url = "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/bowling/-WH-lxmGJVY_000005_000015.mp4"
    command = ["wget", video_url, "-O", sample_video_path]
    subprocess.run(command)
    print("Downloading video")

# Download SSV2 classes if not already present
ssv2_classes_path = "ssv2_classes.json"
if not os.path.exists(ssv2_classes_path):
    command = [
        "wget",
        "https://huggingface.co/datasets/huggingface/label-files/resolve/d79675f2d50a7b1ecf98923d42c30526a51818e2/"
        "something-something-v2-id2label.json",
        "-O",
        "ssv2_classes.json",
    ]
    subprocess.run(command)
    print("Downloading SSV2 classes")

--2026-06-19 16:00:35--  https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/bowling/-WH-lxmGJVY_000005_000015.mp4
Resolving huggingface.co (huggingface.co)... 108.138.246.71, 108.138.246.85, 108.138.246.67, ...
Connecting to huggingface.co (huggingface.co)|108.138.246.71|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/nateraw/kinetics-mini/9f4ed38128a355c352527209101be3e326471816/val%2Fbowling%2F-WH-lxmGJVY_000005_000015.mp4?%2Fdatasets%2Fnateraw%2Fkinetics-mini%2Fresolve%2Fmain%2Fval%2Fbowling%2F-WH-lxmGJVY_000005_000015.mp4=&etag=%2264aa6e6754331c7e567b1b4beaeb48f27cb4bd4f%22 [following]
--2026-06-19 16:00:36--  https://huggingface.co/api/resolve-cache/datasets/nateraw/kinetics-mini/9f4ed38128a355c352527209101be3e326471816/val%2Fbowling%2F-WH-lxmGJVY_000005_000015.mp4?%2Fdatasets%2Fnateraw%2Fkinetics-mini%2Fresolve%2Fmain%2Fval%2Fbowling%2F-WH-lxmGJVY_000005_000015.mp4=&etag=%2264aa6e6754331c7

200 OK
Length: 10103 (9.9K) [text/plain]
Saving to: ‘ssv2_classes.json’

     0K .........                                             100%  177M=0s

2026-06-19 16:00:36 (177 MB/s) - ‘ssv2_classes.json’ saved [10103/10103]



Now, let's load the models in both vanilla Pytorch as well as through the HuggingFace API. Note that HuggingFace API will automatically load the weights through `from_pretrained()`, so there is no additional download required for HuggingFace.

To download the PyTorch model weights, use wget and specify your preferred target path. See the README for the model weight URLs.
E.g. 
```
wget https://dl.fbaipublicfiles.com/vjepa2/vitg-384.pt -P YOUR_DIR
```
Then update `pt_model_path` with `YOUR_DIR/vitg-384.pt`. Also note that you have the option to use `torch.hub.load`.

In [6]:
# HuggingFace model repo name
hf_model_name = (
    "facebook/vjepa2-vitg-fpc64-384"  # Replace with your favored model, e.g. facebook/vjepa2-vitg-fpc64-384
)
# Path to local PyTorch weights
pt_model_path = "../models/vitg-384.pt"

# Initialize the HuggingFace model, load pretrained weights
model_hf = AutoModel.from_pretrained(hf_model_name)
model_hf.cuda().eval()

# Build HuggingFace preprocessing transform
hf_transform = AutoVideoProcessor.from_pretrained(hf_model_name)
img_size = hf_transform.crop_size["height"]

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

In [ ]:
# # HuggingFace model repo name
# hf_model_name = (
#     "facebook/vjepa2-vitg-fpc64-384"  # Replace with your favored model, e.g. facebook/vjepa2-vitg-fpc64-384
# )
# # Path to local PyTorch weights
# pt_model_path = "../models/vitg-384.pt"

# # Initialize the HuggingFace model, auto-place on GPU with most free memory
# model_hf = AutoModel.from_pretrained(hf_model_name)
# device_hf = load_model_to_best_gpu(model_hf, label="HuggingFace model")

# # Build HuggingFace preprocessing transform
# hf_transform = AutoVideoProcessor.from_pretrained(hf_model_name)
# img_size = hf_transform.crop_size["height"]  # E.g. 384, 256, etc.

# Initialize the PyTorch model, auto-place on GPU with most free memory
model_pt = vit_giant_xformers_rope(img_size=(img_size, img_size), num_frames=64)
device_pt = load_model_to_best_gpu(model_pt, label="PyTorch model")
load_pretrained_vjepa_pt_weights(model_pt, pt_model_path)

### Can also use torch.hub to load the model
# model_pt, _ = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant_384')
# device_pt = load_model_to_best_gpu(model_pt, label="PyTorch model (torch.hub)")

# Build PyTorch preprocessing transform
pt_video_transform = build_pt_video_transform(img_size=img_size)

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]


Placing HuggingFace model:
  GPU 0 (Tesla T4): 14.5 GiB free / 14.6 GiB total
  GPU 1 (Tesla T4): 14.5 GiB free / 14.6 GiB total
  GPU 2 (Tesla T4): 14.5 GiB free / 14.6 GiB total
  GPU 3 (Tesla T4): 14.5 GiB free / 14.6 GiB total
  -> Selected cuda:0

Placing PyTorch model:
  GPU 0 (Tesla T4): 10.2 GiB free / 14.6 GiB total
  GPU 1 (Tesla T4): 14.5 GiB free / 14.6 GiB total
  GPU 2 (Tesla T4): 14.5 GiB free / 14.6 GiB total
  GPU 3 (Tesla T4): 14.5 GiB free / 14.6 GiB total
  -> Selected cuda:1
Pretrained weights found at ../models/vitg-384.pt and loaded with msg: <All keys matched successfully>


Now we can run the encoder on the video to get the patch-wise features from the last layer of the encoder. To verify that the HuggingFace and PyTorch models are equivalent, we will compare the values of the features.

In [4]:
def forward_vjepa_video2(model_hf, model_pt, hf_transform, pt_transform):
    # Run a sample inference with VJEPA
    # Automatically detect which device each model is on
    device_hf = next(model_hf.parameters()).device
    device_pt = next(model_pt.parameters()).device
    with torch.inference_mode():
        # Read and pre-process the image
        video = get_video()  # T x H x W x C
        video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
        print(f"Original video shape: {video.shape}")
        x_pt = pt_transform(video).to(device_pt).unsqueeze(0)
        print(f"Transformed video shape for PyTorch model: {x_pt.shape}")
        x_hf = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to(device_hf)
        # Extract the patch-wise features from the last layer
        out_patch_features_pt = model_pt(x_pt)
        out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features_hf, out_patch_features_pt

In [5]:
# Inference on video to get the patch-wise features
out_patch_features_hf, out_patch_features_pt = forward_vjepa_video2(
    model_hf, model_pt, hf_transform, pt_video_transform
)

# Move to CPU for comparison since models are on different GPUs
out_hf_cpu = out_patch_features_hf.cpu()
out_pt_cpu = out_patch_features_pt.cpu()

print(
    f"""
    Inference results on video:
    HuggingFace output shape: {out_hf_cpu.shape}
    PyTorch output shape:     {out_pt_cpu.shape}
    Absolute difference sum:  {torch.abs(out_pt_cpu - out_hf_cpu).sum():.6f}
    Close: {torch.allclose(out_pt_cpu, out_hf_cpu, atol=1e-3, rtol=1e-3)}
    """
)

Original video shape: torch.Size([64, 3, 270, 480])
Transformed video shape for PyTorch model: torch.Size([1, 3, 64, 384, 384])


/home/ubuntu/miniconda3/lib/python3.13/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)



    Inference results on video:
    HuggingFace output shape: torch.Size([1, 18432, 1408])
    PyTorch output shape:     torch.Size([1, 18432, 1408])
    Absolute difference sum:  10294855.000000
    Close: False
    


Great! Now we know that the features from both models are equivalent. Now let's run a pretrained attentive probe classifier on top of the extracted features, to predict an action class for the video. Let's use the Something-Something V2 probe. Note that the repository also includes attentive probe weights for other evaluations such as EPIC-KITCHENS-100 and Diving48.

To download the attentive probe weights, use wget and specify your preferred target path. E.g. `wget https://dl.fbaipublicfiles.com/vjepa2/evals/ssv2-vitg-384-64x2x3.pt -P YOUR_DIR`

Then update `classifier_model_path` with `YOUR_DIR/ssv2-vitg-384-64x2x3.pt`.

In [6]:
# Initialize the classifier (place on same GPU as model_pt since it consumes pt features)
classifier_model_path = "../models/ssv2-vitg-384-64x2x3.pt"
classifier = AttentiveClassifier(
    embed_dim=model_pt.embed_dim, num_heads=16, depth=4, num_classes=174
)
device_cls = next(model_pt.parameters()).device
classifier.to(device_cls).eval()
print(f"Classifier placed on {device_cls} (same as PyTorch model)")
load_pretrained_vjepa_classifier_weights(classifier, classifier_model_path)

# Get classification results
get_vjepa_video_classification_results(classifier, out_patch_features_pt)

Classifier placed on cuda:1 (same as PyTorch model)
Pretrained weights found at ../models/ssv2-vitg-384-64x2x3.pt and loaded with msg: <All keys matched successfully>
Classifier output shape: torch.Size([1, 174])
Top 5 predicted class names:
Putting [something] into [something] (44.9345817565918%)
Stuffing [something] into [something] (28.101837158203125%)
Putting [something] onto [something] (14.43504810333252%)
Failing to put [something] into [something] because [something] does not fit (7.638397693634033%)
Putting [number of] [something] onto [something] (4.890130996704102%)


/tmp/ipykernel_62953/770823340.py:119: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  top5_probs = F.softmax(out_classifier.topk(5).values[0]) * 100.0  # convert to percentage


The video features a man putting a bowling ball into a tube, so the predicted action of "Putting [something] into [something]" makes sense!

This concludes the tutorial. Please see the README and paper for full details on the capabilities of V-JEPA 2 :)